In [41]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import torch

In [43]:
df = pd.read_csv('data.csv')
#df = pd.concat([pd.read_csv('data.csv'), pd.read_csv("random-over-85-from16k.csv")])
df = df.drop_duplicates(subset=["Title"], keep="first")  


In [44]:
# Sample data if you don't have CSV:
# data = {'Title': ['Back to the Future', 'Inception', ...], 
#         'Description': ['A teenager travels back...', 'A thief steals secrets...', ...]}
# df = pd.DataFrame(data)

# Encode movie titles as labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['Title'])
df = df.sort_values(by=["Year", "Title"], ascending=[False, True])

In [45]:
# Load pre-trained SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight but effective

# Generate embeddings for all descriptions
embeddings = model.encode(df['Description'].tolist(), 
                          convert_to_tensor=True,
                          show_progress_bar=True)

# Save embeddings for later reuse
torch.save(embeddings, 'movie_embeddings.pt')
df.to_csv('movie_data_encoded.csv', index=False)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
def predict_movie(description, top_k=3):
    # Encode input description
    input_embedding = model.encode(description, convert_to_tensor=True)
    
    # Find most similar movies
    cos_scores = util.cos_sim(input_embedding, embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k)
    
    # Return results
    print(f"Input: '{description}'\n")
    for score, idx in zip(top_results[0], top_results[1]):
        # Convert tensor to integer using .item() before iloc
        print(f"→ {df.iloc[idx.item()]['Title']} (confidence: {score:.2f})")
    
    # Convert all indices to integers before inverse_transform
    return le.inverse_transform([idx.item() for idx in top_results[1]])

# --- Example Usage ---
test_desc = """Pope Bendict the XVI resigns the Papacy, and he has quite a friendship with his would-be successor Fransicus I"""

predicted = predict_movie(test_desc)

Input: 'The story of the surival and the endavours of a Polish Jewish Pianitst in Warsaw during the events of the German invasion of Poland, his loss of family'

→ The Pianist (confidence: 0.51)
→ All Quiet on the Western Front (confidence: 0.37)
→ Inglourious Basterds (confidence: 0.34)
